# 04 - Continued pre-training, and the price you pay  *(~4 minutes)*

> **Presenter script.** "So you have a model that writes little stories. Your
> boss says: great, now make it write recipes. You do not start from scratch -
> you take the model you have and keep training it on recipes. That's
> **continued pre-training**. It works. It also quietly breaks something, and
> that's the interesting part."

> **continued pre-training** (also *domain-adaptive pre-training*) - take an
> existing checkpoint and keep doing the same next-token training, but on text
> from a new domain.

In [ ]:
# --- boilerplate: make `import minigpt` work no matter where Jupyter started ---
import pathlib
import sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "minigpt").is_dir())
sys.path.insert(0, str(ROOT))

import torch

torch.set_num_threads(4)  # plenty for a model this small; more threads is not faster
print("repo root:", ROOT)

In [ ]:
from minigpt import data
from minigpt import train as T
from minigpt.model import DEFAULT_BATCH_SIZE
from minigpt.plots import plot_bars, plot_two_domains, use_stream_style

import matplotlib.pyplot as plt

use_stream_style()

tokenizer = data.load_tokenizer()

In [ ]:
USE_PREBAKED = True    # False = train both runs live (~2 minutes total)
CPT_STEPS = 400

## 1. Where we're starting from

Load the base checkpoint from notebook 02 and take two measurements we will care
about for the rest of the notebook:

* how well it does on the **old** validation set (stories),
* what it actually writes.

In [ ]:
base_model, _, _ = T.load_checkpoint("base")

old_val = T.encode_to_tensor(data.load_split("base", "val"), tokenizer)
new_val = T.encode_to_tensor(data.load_split("recipes", "val"), tokenizer)
new_train_text = data.load_split("recipes", "train")
old_train_text = data.load_split("base", "train")

before_old = T.evaluate(base_model, old_val, base_model.cfg.block_size, 32, 16, seed=0)
before_new = T.evaluate(base_model, new_val, base_model.cfg.block_size, 32, 16, seed=1)

print(f"base model on OLD domain (stories): {before_old:.3f}   <- it knows this well")
print(f"base model on NEW domain (recipes): {before_new:.3f}   <- it has never seen this")

In [ ]:
T.show_sample(base_model, tokenizer, "one day ", 180, label="BASE MODEL - a story")
T.show_sample(base_model, tokenizer, "recipe:", 180, label="BASE MODEL - asked for a recipe")

## 2. The new domain

Same language, completely different register: numbered steps, imperative voice,
cooking vocabulary. Big enough a jump to make the point.

In [ ]:
print(new_train_text[:420])
print("...")
print(f"\nnew domain: {len(new_train_text):,} characters of training text")

## 3. The naive approach: just keep training

Load the base checkpoint, point it at the recipes, use the same learning rate we
used for pre-training, and go.

While it trains we measure the validation loss on **both** domains at every
checkpoint - the new one it is learning, and the old one it is supposed to
remember.

In [ ]:
if USE_PREBAKED:
    naive = T.load_history("cpt_naive")
    naive_model, _, _ = T.load_checkpoint("cpt_naive")
    print("loaded the pre-baked naive run")
else:
    naive_model, _, _ = T.load_checkpoint("base")
    naive = T.History(name="cpt_naive")
    old_curve = []

    def track_old(step, _tr, _va):
        old_curve.append(
            T.evaluate(naive_model, old_val, naive_model.cfg.block_size, 32, 16, seed=step + 2)
        )

    T.train_model(naive_model, T.encode_to_tensor(new_train_text, tokenizer), new_val,
                  steps=CPT_STEPS, batch_size=DEFAULT_BATCH_SIZE, learning_rate=3e-3,
                  eval_every=50, eval_iters=16, eval_batch_size=32, warmup_steps=20,
                  seed=1337, name="cpt_naive", history=naive, on_eval=track_old)
    naive.meta["old_domain_val"] = old_curve

## 4. The bad news

Plot both validation curves on one chart. Green is the new domain (going well).
Purple is the old domain.

In [ ]:
plot_two_domains(naive.steps, naive.meta["old_domain_val"], naive.val_loss,
                 title="Catastrophic forgetting: it learned recipes and forgot stories")
plt.show()

after_old = naive.meta["old_domain_val"][-1]
print(f"OLD domain loss before: {before_old:.3f}")
print(f"OLD domain loss after : {after_old:.3f}   ({after_old / before_old:.1f}x WORSE)")
print(f"NEW domain loss after : {naive.val_loss[-1]:.3f}   (much better - it did learn recipes)")

> **catastrophic forgetting** - when training on new data destroys what the
> model previously knew. The parameters that encoded "how a story goes" get
> overwritten by "how a recipe goes", because nothing in the training signal
> asked the model to keep them.

It is not subtle. Ask the model for a story now:

In [ ]:
T.show_sample(naive_model, tokenizer, "one day ", 180,
              label="AFTER NAIVE CONTINUED PRE-TRAINING - asked for a story")

It starts a story and slides into a recipe within a few words. It has not been
"damaged" - it has been *overwritten*.

In [ ]:
plot_bars(["before (stories)", "after (stories)"], [before_old, after_old],
          title="What continued pre-training did to the ORIGINAL domain",
          ylabel="validation loss on the OLD domain",
          colors=["#9467bd", "#d62728"])
plt.show()

## 5. The fix: go gently, and keep revising the old material

Two changes, both cheap:

1. **Lower the learning rate** (we drop it 6x, from `0.003` to `0.0005`). Smaller
   nudges move the model towards recipes without bulldozing what is already
   there.
2. **Replay** (sometimes *rehearsal*): mix ~30% of the *original* corpus back
   into the new training data. The model keeps being reminded of the old domain
   while it learns the new one.

That is it. That is the whole mitigation, and it is what real labs do too - at
much larger scale, with much more careful mixing ratios.

In [ ]:
mixed_text = T.mix_texts(new_train_text, old_train_text, replay_fraction=0.3, seed=1337)
print(f"recipes only : {len(new_train_text):,} characters")
print(f"with replay  : {len(mixed_text):,} characters "
      f"({(len(mixed_text) - len(new_train_text)) / len(mixed_text):.0%} of it is the old corpus)")

In [ ]:
if USE_PREBAKED:
    replay = T.load_history("cpt_replay")
    replay_model, _, _ = T.load_checkpoint("cpt_replay")
    print("loaded the pre-baked replay run")
else:
    replay_model, _, _ = T.load_checkpoint("base")
    replay = T.History(name="cpt_replay")
    old_curve = []

    def track_old_replay(step, _tr, _va):
        old_curve.append(
            T.evaluate(replay_model, old_val, replay_model.cfg.block_size, 32, 16, seed=step + 2)
        )

    T.train_model(replay_model, T.encode_to_tensor(mixed_text, tokenizer), new_val,
                  steps=CPT_STEPS, batch_size=DEFAULT_BATCH_SIZE, learning_rate=5e-4,
                  eval_every=50, eval_iters=16, eval_batch_size=32, warmup_steps=20,
                  seed=1337, name="cpt_replay", history=replay, on_eval=track_old_replay)
    replay.meta["old_domain_val"] = old_curve

In [ ]:
plot_two_domains(replay.steps, replay.meta["old_domain_val"], replay.val_loss,
                 title="Lower learning rate + 30% replay: it learns AND remembers")
plt.show()

## 6. Side by side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for ax, (hist, label) in zip(axes, [(naive, "NAIVE\nhigh LR, new data only"),
                                    (replay, "MITIGATED\nlow LR + 30% replay")]):
    ax.plot(hist.steps, hist.meta["old_domain_val"], "-o", color="#9467bd",
            label="OLD domain (stories)")
    ax.plot(hist.steps, hist.val_loss, "-o", color="#2ca02c", label="NEW domain (recipes)")
    ax.axhline(before_old, ls=":", color="grey", label="where the old domain started")
    ax.set_title(label)
    ax.set_xlabel("continued pre-training step")
    ax.legend(fontsize=9)
axes[0].set_ylabel("validation loss")
fig.tight_layout()
plt.show()

In [ ]:
rows = [
    ("old domain (stories)", before_old, naive.meta["old_domain_val"][-1],
     replay.meta["old_domain_val"][-1]),
    ("new domain (recipes)", before_new, naive.val_loss[-1], replay.val_loss[-1]),
]
print(f"{'validation loss':<24}{'base':>10}{'naive':>10}{'replay':>10}")
print("-" * 54)
for name, base_v, naive_v, replay_v in rows:
    print(f"{name:<24}{base_v:>10.3f}{naive_v:>10.3f}{replay_v:>10.3f}")
print()
print("Naive wins on recipes. Replay is only a bit behind on recipes,")
print("and it is dramatically better at not destroying the original model.")

In [ ]:
T.show_sample(replay_model, tokenizer, "one day ", 160, label="REPLAY MODEL - asked for a story")
T.show_sample(replay_model, tokenizer, "recipe:", 160, label="REPLAY MODEL - asked for a recipe")

## Recap

* Continued pre-training = keep doing next-token training on a new domain.
* It works, and by default it **erases** the old domain. That is catastrophic
  forgetting, and you will only notice it if you keep evaluating on the *old*
  validation set.
* **Always keep the old validation set.** It is your smoke alarm.
* The standard mitigations are boring and effective: **lower learning rate** and
  **replay some original data**.
* There is always a trade-off. Replay costs you a little on the new domain. You
  choose where on that line you want to sit.

**Next:** `05_posttraining.ipynb` - turning a text continuation engine into
something that answers questions.